# Local code knowledge graph

This notebook builds an Oracle-free code knowledge graph for
`/home/user/projects/implicit-decision-gate`.

It implements the reusable parts of the L4 notebook explicitly:

- Tracked source files and Python symbols become typed nodes.
- Containment, imports, calls, instantiation, inheritance, tests, and Git co-change
  evidence become weighted edges.
- Word and identifier-aware TF-IDF finds query anchors without an external model.
- Weighted Personalized PageRank and strongest-path expansion find related nodes.
- Plotly renders portable file-level and query-specific interactive graphs.

The target repository is only read. Its modules aren't imported or executed.

Launch from this directory with:

```bash
UV_CACHE_DIR=/tmp/code-knowledge-graph-uv-cache uv sync --group dev
UV_CACHE_DIR=/tmp/code-knowledge-graph-uv-cache uv run jupyter lab \
    knowledge_code_graph.ipynb
```

In [12]:
from __future__ import annotations

import ast
import html
import importlib.util
import itertools
import math
import os
import re
import subprocess
import tokenize
from collections import Counter, defaultdict
from collections.abc import Iterable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, cast

import networkx as nx
import pandas as pd
import plotly.graph_objects as go
from IPython import get_ipython
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import FeatureUnion


def show(value: object) -> None:
    display(value)  # type: ignore[no-untyped-call]


def html_block(value: str) -> object:
    return HTML(value)  # type: ignore[no-untyped-call]


TARGET_REPO = (
    Path(
        os.environ.get(
            "CODE_GRAPH_TARGET",
            "/home/user/projects/implicit-decision-gate",
        )
    )
    .expanduser()
    .resolve()
)
ARTIFACT_DIR = (Path.cwd() / "artifacts").resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

if not TARGET_REPO.is_dir():
    raise FileNotFoundError(f"Target repository doesn't exist: {TARGET_REPO}")

print(f"Target: {TARGET_REPO}")

Target: /home/user/projects/implicit-decision-gate


## Graph records

IDs are stable across runs as long as a path and qualified name don't change. Edge
strength is always in the inclusive range from `0.0` to `1.0`. Confidence records how
reliable extraction was, while strength also includes the relationship-type prior and
repeated evidence.

In [13]:
@dataclass(frozen=True)
class CodeNode:
    id: str
    kind: str
    name: str
    qualified_name: str
    path: str
    module: str
    language: str
    start_line: int
    end_line: int
    signature: str = ""
    docstring: str = ""
    source: str = ""

    @property
    def location(self) -> str:
        if self.start_line <= 0:
            return self.path
        return f"{self.path}:{self.start_line}"


@dataclass(frozen=True)
class RawRelation:
    source_id: str
    target_id: str
    kind: str
    confidence: float
    evidence: str
    count: int = 1


@dataclass(frozen=True)
class PendingRelation:
    source_id: str
    target_text: str
    kind: str
    confidence: float
    evidence: str


@dataclass(frozen=True)
class ImportBinding:
    kind: str
    module: str
    symbol: str = ""


@dataclass(frozen=True)
class CodeEdge:
    source_id: str
    target_id: str
    kind: str
    count: int
    confidence: float
    strength: float
    evidence: tuple[str, ...]


@dataclass(frozen=True)
class ParseIssue:
    path: str
    message: str


@dataclass
class ExtractionState:
    nodes: dict[str, CodeNode] = field(default_factory=dict)
    raw_relations: list[RawRelation] = field(default_factory=list)
    pending_relations: list[PendingRelation] = field(default_factory=list)
    aliases_by_path: dict[str, dict[str, ImportBinding]] = field(default_factory=dict)
    issues: list[ParseIssue] = field(default_factory=list)


@dataclass(frozen=True)
class KnowledgeGraph:
    repository: Path
    nodes: Mapping[str, CodeNode]
    edges: tuple[CodeEdge, ...]
    graph: nx.MultiDiGraph[str]
    issues: tuple[ParseIssue, ...]
    resolution_counts: Mapping[str, int]


@dataclass(frozen=True)
class SearchIndex:
    node_ids: tuple[str, ...]
    vectorizer: FeatureUnion
    matrix: Any


@dataclass(frozen=True)
class QueryResult:
    query: str
    anchors: tuple[str, ...]
    relevant: pd.DataFrame
    related: pd.DataFrame
    direct_scores: Mapping[str, float]
    relationship_scores: Mapping[str, float]
    pagerank_scores: Mapping[str, float]
    paths: Mapping[str, tuple[str, ...]]


EDGE_PRIORS: Mapping[str, float] = {
    "CALLS": 1.00,
    "INHERITS": 0.95,
    "INSTANTIATES": 0.90,
    "TESTS": 0.85,
    "CO_CHANGES": 0.80,
    "IMPORTS": 0.65,
    "CONTAINS": 0.45,
}

## Discover source files and parse Python symbols

Git supplies tracked and untracked, non-ignored Python paths. The fallback excludes
common generated and environment directories. Non-Python code is searchable at file
level. `tokenize.open` respects source encoding, and `ast.parse` analyzes Python without
running it.

In [14]:
EXCLUDED_DIRECTORIES = {
    ".git",
    ".idg",
    ".mypy_cache",
    ".pytest_cache",
    ".ruff_cache",
    ".venv",
    "__pycache__",
    "build",
    "dist",
    "node_modules",
}
SOURCE_SUFFIXES = {
    ".css",
    ".html",
    ".js",
    ".json",
    ".jsx",
    ".mjs",
    ".py",
    ".sql",
    ".toml",
    ".ts",
    ".tsx",
    ".yaml",
    ".yml",
}
LANGUAGE_BY_SUFFIX: Mapping[str, str] = {
    ".css": "css",
    ".html": "html",
    ".js": "javascript",
    ".json": "json",
    ".jsx": "javascript",
    ".mjs": "javascript",
    ".py": "python",
    ".sql": "sql",
    ".toml": "toml",
    ".ts": "typescript",
    ".tsx": "typescript",
    ".yaml": "yaml",
    ".yml": "yaml",
}


def discover_source_files(repository: Path) -> tuple[Path, ...]:
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(repository),
            "ls-files",
            "-co",
            "--exclude-standard",
            "-z",
        ],
        check=False,
        capture_output=True,
        text=False,
    )
    if completed.returncode == 0:
        relative_paths = [
            Path(item.decode("utf-8", errors="surrogateescape"))
            for item in completed.stdout.split(b"\0")
            if item
        ]
        return tuple(
            sorted(
                repository / path
                for path in relative_paths
                if path.suffix.lower() in SOURCE_SUFFIXES
                and not any(part in EXCLUDED_DIRECTORIES for part in path.parts)
            )
        )

    return tuple(
        sorted(
            path
            for path in repository.rglob("*")
            if path.is_file()
            and path.suffix.lower() in SOURCE_SUFFIXES
            and not any(part in EXCLUDED_DIRECTORIES for part in path.relative_to(repository).parts)
        )
    )


def module_name_for_path(relative_path: Path) -> str:
    parts = list(relative_path.with_suffix("").parts)
    if parts and parts[0] == "src":
        parts = parts[1:]
    if parts and parts[-1] == "__init__":
        parts = parts[:-1]
    return ".".join(parts) or relative_path.stem


def dotted_name(expression: ast.expr) -> str | None:
    if isinstance(expression, ast.Name):
        return expression.id
    if isinstance(expression, ast.Attribute):
        parent = dotted_name(expression.value)
        return f"{parent}.{expression.attr}" if parent else expression.attr
    return None


def function_signature(node: ast.FunctionDef | ast.AsyncFunctionDef) -> str:
    prefix = "async " if isinstance(node, ast.AsyncFunctionDef) else ""
    returns = f" -> {ast.unparse(node.returns)}" if node.returns is not None else ""
    return f"{prefix}{node.name}({ast.unparse(node.args)}){returns}"


def class_signature(node: ast.ClassDef) -> str:
    bases = ", ".join(ast.unparse(base) for base in node.bases)
    return f"class {node.name}({bases})" if bases else f"class {node.name}"


def resolve_relative_module(
    current_module: str,
    relative_path: Path,
    imported_module: str | None,
    level: int,
) -> str:
    if level == 0:
        return imported_module or ""
    package = (
        current_module if relative_path.stem == "__init__" else current_module.rpartition(".")[0]
    )
    request = "." * level + (imported_module or "")
    if not package:
        return imported_module or ""
    try:
        return importlib.util.resolve_name(request, package)
    except (ImportError, ValueError):
        return imported_module or ""


class PythonFileExtractor(ast.NodeVisitor):
    def __init__(
        self,
        *,
        repository: Path,
        path: Path,
        source: str,
        state: ExtractionState,
    ) -> None:
        self.repository = repository
        self.path = path
        self.relative_path = path.relative_to(repository)
        self.path_text = self.relative_path.as_posix()
        self.source = source
        self.lines = source.splitlines()
        self.state = state
        self.module = module_name_for_path(self.relative_path)
        self.file_id = f"file:{self.path_text}"
        self.scope: list[tuple[str, str, str]] = []
        self.aliases: dict[str, ImportBinding] = {}

    def extract(self, tree: ast.Module) -> None:
        self.state.nodes[self.file_id] = CodeNode(
            id=self.file_id,
            kind="file",
            name=self.relative_path.name,
            qualified_name=self.module,
            path=self.path_text,
            module=self.module,
            language="python",
            start_line=1,
            end_line=max(1, len(self.lines)),
            docstring=ast.get_docstring(tree, clean=True) or "",
            source=self.source,
        )
        self.visit(tree)
        self.state.aliases_by_path[self.path_text] = self.aliases

    @property
    def current_source_id(self) -> str:
        return self.scope[-1][2] if self.scope else self.file_id

    @property
    def current_qualified_name(self) -> str:
        return ".".join(name for name, _kind, _node_id in self.scope)

    def source_segment(self, node: ast.AST) -> str:
        start = max(1, getattr(node, "lineno", 1))
        end = max(start, getattr(node, "end_lineno", start))
        return "\n".join(self.lines[start - 1 : end])

    def add_symbol(
        self,
        node: ast.ClassDef | ast.FunctionDef | ast.AsyncFunctionDef,
        kind: str,
        signature: str,
    ) -> tuple[str, str]:
        parent_id = self.current_source_id
        qualified_name = ".".join(
            [part for part in (self.current_qualified_name, node.name) if part]
        )
        node_id = f"symbol:{self.path_text}::{qualified_name}"
        start = getattr(node, "lineno", 1)
        end = getattr(node, "end_lineno", start)
        self.state.nodes[node_id] = CodeNode(
            id=node_id,
            kind=kind,
            name=node.name,
            qualified_name=qualified_name,
            path=self.path_text,
            module=self.module,
            language="python",
            start_line=start,
            end_line=end,
            signature=signature,
            docstring=ast.get_docstring(node, clean=True) or "",
            source=self.source_segment(node),
        )
        self.state.raw_relations.append(
            RawRelation(
                source_id=parent_id,
                target_id=node_id,
                kind="CONTAINS",
                confidence=1.0,
                evidence=f"{self.path_text}:{start}",
            )
        )
        return node_id, qualified_name

    def visit_ClassDef(self, node: ast.ClassDef) -> None:
        node_id, _qualified_name = self.add_symbol(node, "class", class_signature(node))
        for base in node.bases:
            target = dotted_name(base)
            if target:
                self.state.pending_relations.append(
                    PendingRelation(
                        source_id=node_id,
                        target_text=target,
                        kind="INHERITS",
                        confidence=1.0,
                        evidence=f"{self.path_text}:{node.lineno}",
                    )
                )
        self.scope.append((node.name, "class", node_id))
        self.generic_visit(node)
        self.scope.pop()

    def visit_FunctionDef(self, node: ast.FunctionDef) -> None:
        self._visit_function(node)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef) -> None:
        self._visit_function(node)

    def _visit_function(self, node: ast.FunctionDef | ast.AsyncFunctionDef) -> None:
        parent_kind = self.scope[-1][1] if self.scope else ""
        is_test = self.path_text.startswith("tests/") and node.name.startswith("test_")
        decorator_names = {dotted_name(decorator) for decorator in node.decorator_list}
        is_fixture = any(name and name.endswith("fixture") for name in decorator_names)
        if is_test:
            kind = "test"
        elif is_fixture:
            kind = "fixture"
        elif parent_kind == "class":
            kind = "method"
        else:
            kind = "function"
        node_id, _qualified_name = self.add_symbol(node, kind, function_signature(node))
        self.scope.append((node.name, kind, node_id))
        self.generic_visit(node)
        self.scope.pop()

    def visit_Import(self, node: ast.Import) -> None:
        for alias in node.names:
            bound_name = alias.asname or alias.name.split(".")[0]
            bound_module = alias.name if alias.asname else alias.name.split(".")[0]
            self.aliases[bound_name] = ImportBinding("module", bound_module)
            self.state.pending_relations.append(
                PendingRelation(
                    source_id=self.file_id,
                    target_text=f"@module:{alias.name}",
                    kind="IMPORTS",
                    confidence=1.0,
                    evidence=f"{self.path_text}:{node.lineno}",
                )
            )

    def visit_ImportFrom(self, node: ast.ImportFrom) -> None:
        module = resolve_relative_module(
            self.module,
            self.relative_path,
            node.module,
            node.level,
        )
        if module:
            self.state.pending_relations.append(
                PendingRelation(
                    source_id=self.file_id,
                    target_text=f"@module:{module}",
                    kind="IMPORTS",
                    confidence=1.0,
                    evidence=f"{self.path_text}:{node.lineno}",
                )
            )
        for alias in node.names:
            if alias.name == "*":
                continue
            bound_name = alias.asname or alias.name
            self.aliases[bound_name] = ImportBinding("symbol", module, alias.name)

    def visit_Call(self, node: ast.Call) -> None:
        target = dotted_name(node.func)
        if target:
            self.state.pending_relations.append(
                PendingRelation(
                    source_id=self.current_source_id,
                    target_text=target,
                    kind="CALLS",
                    confidence=1.0,
                    evidence=f"{self.path_text}:{node.lineno}",
                )
            )
        self.generic_visit(node)


def parse_repository(repository: Path) -> ExtractionState:
    state = ExtractionState()
    for path in discover_source_files(repository):
        relative_path = path.relative_to(repository).as_posix()
        try:
            with tokenize.open(path) as stream:
                source = stream.read()
            if path.suffix.lower() == ".py":
                tree = ast.parse(source, filename=relative_path, type_comments=True)
                PythonFileExtractor(
                    repository=repository,
                    path=path,
                    source=source,
                    state=state,
                ).extract(tree)
            else:
                relative = path.relative_to(repository)
                path_text = relative.as_posix()
                file_id = f"file:{path_text}"
                state.nodes[file_id] = CodeNode(
                    id=file_id,
                    kind="file",
                    name=relative.name,
                    qualified_name=module_name_for_path(relative),
                    path=path_text,
                    module=module_name_for_path(relative),
                    language=LANGUAGE_BY_SUFFIX.get(path.suffix.lower(), "text"),
                    start_line=1,
                    end_line=max(1, len(source.splitlines())),
                    source=source,
                )
        except (OSError, SyntaxError, UnicodeError) as exc:
            state.issues.append(ParseIssue(relative_path, str(exc)))
    return state

## Resolve symbols and add Git co-change evidence

Calls are resolved in this order: `self` or `cls`, imported aliases, same-module
symbols, fully qualified module paths, then a unique repository-wide short name. The
last case has lower confidence. Unresolved dynamic calls aren't converted into edges.

A co-change strength uses commit-level Jaccard similarity and support. Merge commits and
very broad changes are excluded because they provide weak evidence of a direct relation.

In [15]:
def module_and_symbol_indexes(
    nodes: Mapping[str, CodeNode],
) -> tuple[
    dict[str, str],
    dict[tuple[str, str], str],
    dict[tuple[str, str], list[str]],
    dict[str, list[str]],
]:
    module_to_file = {node.module: node.id for node in nodes.values() if node.kind == "file"}
    exact_symbols: dict[tuple[str, str], str] = {}
    module_short_names: dict[tuple[str, str], list[str]] = defaultdict(list)
    global_short_names: dict[str, list[str]] = defaultdict(list)
    for node in nodes.values():
        if node.kind == "file":
            continue
        exact_symbols[(node.module, node.qualified_name)] = node.id
        module_short_names[(node.module, node.name)].append(node.id)
        global_short_names[node.name].append(node.id)
    return module_to_file, exact_symbols, module_short_names, global_short_names


def enclosing_class_name(
    source: CodeNode,
    exact_symbols: Mapping[tuple[str, str], str],
    nodes: Mapping[str, CodeNode],
) -> str | None:
    parts = source.qualified_name.split(".")
    for end in range(len(parts) - 1, 0, -1):
        candidate_name = ".".join(parts[:end])
        candidate_id = exact_symbols.get((source.module, candidate_name))
        if candidate_id and nodes[candidate_id].kind == "class":
            return candidate_name
    return None


def resolve_symbol_text(
    *,
    source: CodeNode,
    target_text: str,
    nodes: Mapping[str, CodeNode],
    aliases: Mapping[str, ImportBinding],
    module_to_file: Mapping[str, str],
    exact_symbols: Mapping[tuple[str, str], str],
    module_short_names: Mapping[tuple[str, str], list[str]],
    global_short_names: Mapping[str, list[str]],
) -> tuple[str, float] | None:
    if target_text.startswith("@module:"):
        module = target_text.removeprefix("@module:")
        target_id = module_to_file.get(module)
        if target_id:
            return target_id, 1.0
        return None

    parts = target_text.split(".")
    root = parts[0]
    remainder = ".".join(parts[1:])

    if root in {"self", "cls"} and remainder:
        class_name = enclosing_class_name(source, exact_symbols, nodes)
        if class_name:
            target_id = exact_symbols.get((source.module, f"{class_name}.{remainder}"))
            if target_id:
                return target_id, 0.95

    binding = aliases.get(root)
    if binding:
        if binding.kind == "module":
            candidate_module = binding.module
            candidate_symbol = remainder
        else:
            candidate_module = binding.module
            candidate_symbol = ".".join(part for part in (binding.symbol, remainder) if part)
        if candidate_symbol:
            target_id = exact_symbols.get((candidate_module, candidate_symbol))
            if target_id:
                return target_id, 0.95
            candidates = module_short_names.get(
                (candidate_module, candidate_symbol.rsplit(".", 1)[-1]),
                [],
            )
            if len(candidates) == 1:
                return candidates[0], 0.85
        elif candidate_module in module_to_file:
            return module_to_file[candidate_module], 0.95

    if "." in target_text:
        target_id = exact_symbols.get((source.module, target_text))
        if target_id:
            return target_id, 0.95
    else:
        class_name = enclosing_class_name(source, exact_symbols, nodes)
        if class_name:
            target_id = exact_symbols.get((source.module, f"{class_name}.{target_text}"))
            if target_id:
                return target_id, 0.95
        candidates = module_short_names.get((source.module, target_text), [])
        top_level = [
            candidate for candidate in candidates if "." not in nodes[candidate].qualified_name
        ]
        if len(top_level) == 1:
            return top_level[0], 0.95
        if len(candidates) == 1:
            return candidates[0], 0.85

    for module in sorted(module_to_file, key=len, reverse=True):
        prefix = f"{module}."
        if target_text.startswith(prefix):
            target_id = exact_symbols.get((module, target_text.removeprefix(prefix)))
            if target_id:
                return target_id, 0.90

    candidates = global_short_names.get(parts[-1], [])
    if len(candidates) == 1:
        return candidates[0], 0.60
    return None


def resolve_pending_relations(
    state: ExtractionState,
) -> tuple[list[RawRelation], Counter[str]]:
    module_to_file, exact_symbols, module_short_names, global_short_names = (
        module_and_symbol_indexes(state.nodes)
    )
    resolved = list(state.raw_relations)
    counts: Counter[str] = Counter()
    for relation in state.pending_relations:
        source = state.nodes[relation.source_id]
        aliases = state.aliases_by_path.get(source.path, {})
        match = resolve_symbol_text(
            source=source,
            target_text=relation.target_text,
            nodes=state.nodes,
            aliases=aliases,
            module_to_file=module_to_file,
            exact_symbols=exact_symbols,
            module_short_names=module_short_names,
            global_short_names=global_short_names,
        )
        if match is None:
            label = (
                "external_imports"
                if relation.kind == "IMPORTS"
                else (f"unresolved_{relation.kind.lower()}")
            )
            counts[label] += 1
            continue
        target_id, resolution_confidence = match
        if target_id == relation.source_id:
            counts[f"self_{relation.kind.lower()}"] += 1
            continue
        target = state.nodes[target_id]
        kind = relation.kind
        if kind == "CALLS" and target.kind == "class":
            kind = "INSTANTIATES"
        elif kind == "CALLS" and source.kind == "test" and not target.path.startswith("tests/"):
            kind = "TESTS"
        resolved.append(
            RawRelation(
                source_id=relation.source_id,
                target_id=target_id,
                kind=kind,
                confidence=relation.confidence * resolution_confidence,
                evidence=relation.evidence,
            )
        )
        counts[f"resolved_{kind.lower()}"] += 1
    return resolved, counts


def git_cochange_relations(
    repository: Path,
    nodes: Mapping[str, CodeNode],
    *,
    minimum_support: int = 2,
    maximum_files_per_commit: int = 20,
) -> list[RawRelation]:
    file_ids = {node.path: node.id for node in nodes.values() if node.kind == "file"}
    completed = subprocess.run(
        [
            "git",
            "-C",
            str(repository),
            "log",
            "--no-merges",
            "--format=@@%H",
            "--name-only",
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        return []

    commits: list[tuple[str, tuple[str, ...]]] = []
    commit_hash = ""
    changed_paths: set[str] = set()

    def finish_commit() -> None:
        if commit_hash and changed_paths:
            commits.append((commit_hash, tuple(sorted(changed_paths))))

    for line in completed.stdout.splitlines():
        if line.startswith("@@"):
            finish_commit()
            commit_hash = line.removeprefix("@@")
            changed_paths = set()
        elif line and line in file_ids:
            changed_paths.add(line)
    finish_commit()

    file_commit_counts: Counter[str] = Counter()
    pair_counts: Counter[tuple[str, str]] = Counter()
    pair_commits: dict[tuple[str, str], list[str]] = defaultdict(list)
    for current_hash, paths in commits:
        file_commit_counts.update(paths)
        if len(paths) > maximum_files_per_commit:
            continue
        for pair in itertools.combinations(paths, 2):
            pair_counts[pair] += 1
            pair_commits[pair].append(current_hash)

    relations: list[RawRelation] = []
    for (left, right), count in pair_counts.items():
        if count < minimum_support:
            continue
        union_count = file_commit_counts[left] + file_commit_counts[right] - count
        jaccard = count / union_count if union_count else 0.0
        support = 1.0 - math.exp(-count / 2.0)
        confidence = min(1.0, jaccard * support)
        example_commits = ", ".join(value[:8] for value in pair_commits[(left, right)])
        relations.append(
            RawRelation(
                source_id=file_ids[left],
                target_id=file_ids[right],
                kind="CO_CHANGES",
                confidence=confidence,
                evidence=(
                    f"{count} non-merge commits; Jaccard={jaccard:.3f}; commits={example_commits}"
                ),
                count=count,
            )
        )
    return relations

## Aggregate evidence and construct the graph

Repeated observations of the same typed relation become one edge. For relationship type
`t`, confidence `c`, occurrence count `n`, and maximum count `n_max` within that type,
the edge strength is:

`prior[t] * c * (0.5 + 0.5 * log(1 + n) / log(1 + n_max))`

The type prior prevents containment and imports from dominating direct calls and
inheritance. All values and evidence remain visible in query results and tooltips.

In [16]:
def aggregate_relations(relations: Iterable[RawRelation]) -> tuple[CodeEdge, ...]:
    grouped: dict[tuple[str, str, str], dict[str, Any]] = {}
    for relation in relations:
        key = (relation.source_id, relation.target_id, relation.kind)
        group = grouped.setdefault(
            key,
            {
                "count": 0,
                "weighted_confidence": 0.0,
                "evidence": set(),
            },
        )
        group["count"] += relation.count
        group["weighted_confidence"] += relation.confidence * relation.count
        group["evidence"].add(relation.evidence)

    maximum_count_by_kind: Counter[str] = Counter()
    for (_source, _target, kind), group in grouped.items():
        maximum_count_by_kind[kind] = max(maximum_count_by_kind[kind], group["count"])

    edges: list[CodeEdge] = []
    for (source_id, target_id, kind), group in sorted(grouped.items()):
        count = int(group["count"])
        confidence = float(group["weighted_confidence"]) / count
        maximum_count = maximum_count_by_kind[kind]
        frequency = math.log1p(count) / math.log1p(maximum_count)
        prior = EDGE_PRIORS.get(kind, 0.50)
        strength = min(1.0, prior * confidence * (0.5 + 0.5 * frequency))
        edges.append(
            CodeEdge(
                source_id=source_id,
                target_id=target_id,
                kind=kind,
                count=count,
                confidence=confidence,
                strength=strength,
                evidence=tuple(sorted(group["evidence"])),
            )
        )
    return tuple(edges)


def create_networkx_graph(
    nodes: Mapping[str, CodeNode],
    edges: Sequence[CodeEdge],
) -> nx.MultiDiGraph[str]:
    graph: nx.MultiDiGraph[str] = nx.MultiDiGraph()
    for node in nodes.values():
        graph.add_node(node.id, **vars(node))
    for edge in edges:
        graph.add_edge(
            edge.source_id,
            edge.target_id,
            key=edge.kind,
            **vars(edge),
        )
    return graph


def validate_knowledge_graph(
    nodes: Mapping[str, CodeNode],
    edges: Sequence[CodeEdge],
) -> None:
    missing_endpoints = [
        edge for edge in edges if edge.source_id not in nodes or edge.target_id not in nodes
    ]
    invalid_strengths = [edge for edge in edges if not 0.0 <= edge.strength <= 1.0]
    invalid_spans = [
        node for node in nodes.values() if node.start_line < 1 or node.end_line < node.start_line
    ]
    if missing_endpoints:
        raise ValueError(f"Edges with missing endpoints: {missing_endpoints[:3]}")
    if invalid_strengths:
        raise ValueError(f"Edges with invalid strengths: {invalid_strengths[:3]}")
    if invalid_spans:
        raise ValueError(f"Nodes with invalid source spans: {invalid_spans[:3]}")


def build_knowledge_graph(repository: Path) -> KnowledgeGraph:
    state = parse_repository(repository)
    resolved, resolution_counts = resolve_pending_relations(state)
    resolved.extend(git_cochange_relations(repository, state.nodes))
    edges = aggregate_relations(resolved)
    validate_knowledge_graph(state.nodes, edges)
    graph = create_networkx_graph(state.nodes, edges)
    return KnowledgeGraph(
        repository=repository,
        nodes=dict(state.nodes),
        edges=edges,
        graph=graph,
        issues=tuple(state.issues),
        resolution_counts=dict(resolution_counts),
    )


def graph_summary(knowledge: KnowledgeGraph) -> tuple[pd.DataFrame, pd.DataFrame]:
    node_counts = Counter(node.kind for node in knowledge.nodes.values())
    edge_counts = Counter(edge.kind for edge in knowledge.edges)
    node_frame = pd.DataFrame(
        [{"node_kind": kind, "count": count} for kind, count in node_counts.most_common()]
    )
    edge_frame = pd.DataFrame(
        [
            {
                "edge_kind": kind,
                "count": count,
                "mean_strength": round(
                    sum(edge.strength for edge in knowledge.edges if edge.kind == kind) / count,
                    3,
                ),
            }
            for kind, count in edge_counts.most_common()
        ]
    )
    return node_frame, edge_frame


knowledge = build_knowledge_graph(TARGET_REPO)
node_summary, edge_summary = graph_summary(knowledge)

print(
    f"Built {len(knowledge.nodes)} nodes and {len(knowledge.edges)} typed edges "
    f"from {sum(1 for node in knowledge.nodes.values() if node.kind == 'file')} files."
)
show(node_summary)
show(edge_summary)
show(pd.DataFrame([knowledge.resolution_counts]))
if knowledge.issues:
    show(pd.DataFrame([vars(issue) for issue in knowledge.issues]))

Built 242 nodes and 879 typed edges from 34 files.


,node_kind,count
0,function,66
1,method,59
2,class,41
3,test,41
4,file,34
5,fixture,1


,edge_kind,count,mean_strength
0,CO_CHANGES,212,0.223
1,CONTAINS,208,0.450
2,CALLS,157,0.611
3,INSTANTIATES,156,0.595
4,IMPORTS,88,0.650
5,TESTS,57,0.445
6,INHERITS,1,0.902


,external_imports,resolved_imports,unresolved_calls,resolved_instantiates,resolved_calls,unresolved_inherits,resolved_tests,resolved_inherits
0,122,88,564,241,203,19,70,1


## Build the local search index

The index combines word unigrams and bigrams with character n-grams. Character features
make `RunStore`, `run_store`, and partial identifiers match without a hosted embedding
model. Node text includes the path, qualified name, signature, docstring, and source.

In [17]:
CAMEL_BOUNDARY = re.compile(r"(?<=[a-z0-9])(?=[A-Z])")


def expand_identifiers(value: str) -> str:
    expanded = CAMEL_BOUNDARY.sub(" ", value)
    return expanded.replace("_", " ").replace("-", " ")


def node_document(node: CodeNode) -> str:
    identity = " ".join(
        [
            node.kind,
            node.path,
            node.module,
            node.language,
            node.qualified_name,
            node.signature,
            node.docstring,
        ]
    )
    return f"{identity}\n{expand_identifiers(identity)}\n{node.source}"


def build_search_index(nodes: Mapping[str, CodeNode]) -> SearchIndex:
    node_ids = tuple(sorted(nodes))
    documents = [node_document(nodes[node_id]) for node_id in node_ids]
    vectorizer = FeatureUnion(
        [
            (
                "word",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 2),
                    sublinear_tf=True,
                    token_pattern=r"(?u)\b[\w.]+\b",
                ),
            ),
            (
                "identifier",
                TfidfVectorizer(
                    analyzer="char_wb",
                    lowercase=True,
                    ngram_range=(3, 5),
                    sublinear_tf=True,
                ),
            ),
        ],
        transformer_weights={"word": 0.70, "identifier": 0.30},
    )
    matrix = vectorizer.fit_transform(documents)
    return SearchIndex(node_ids, vectorizer, matrix)


def direct_query_scores(index: SearchIndex, query: str) -> dict[str, float]:
    query_vector = index.vectorizer.transform([expand_identifiers(query)])
    values = cosine_similarity(query_vector, index.matrix).ravel()
    return {node_id: float(value) for node_id, value in zip(index.node_ids, values, strict=True)}


search_index = build_search_index(knowledge.nodes)
print(f"Indexed {len(search_index.node_ids)} node documents.")

Indexed 242 node documents.


## Query relevant and related nodes

Directly relevant nodes are the highest TF-IDF matches. They seed weighted Personalized
PageRank. A separate bounded expansion finds each related node's strongest path from a
seed. The related table exposes both scores and is ordered by relationship strength.
Traversal treats code relations as bidirectional for discovery, while the displayed path
preserves the original edge direction.

In [18]:
def relationship_projection(knowledge: KnowledgeGraph) -> nx.Graph[str]:
    projection: nx.Graph[str] = nx.Graph()
    projection.add_nodes_from(knowledge.nodes)
    for edge in knowledge.edges:
        if projection.has_edge(edge.source_id, edge.target_id):
            current = float(projection[edge.source_id][edge.target_id]["weight"])
            combined = 1.0 - (1.0 - current) * (1.0 - edge.strength)
            projection[edge.source_id][edge.target_id]["weight"] = combined
            projection[edge.source_id][edge.target_id]["kinds"].add(edge.kind)
        else:
            projection.add_edge(
                edge.source_id,
                edge.target_id,
                weight=edge.strength,
                kinds={edge.kind},
            )
    return projection


def strongest_paths(
    projection: nx.Graph[str],
    seed_scores: Mapping[str, float],
    *,
    hops: int,
    hop_decay: float,
) -> tuple[dict[str, float], dict[str, tuple[str, ...]]]:
    maximum_seed_score = max(seed_scores.values(), default=1.0) or 1.0
    best_scores: dict[str, float] = {}
    best_paths: dict[str, tuple[str, ...]] = {}
    frontier: list[tuple[str, float, tuple[str, ...]]] = [
        (seed_id, score / maximum_seed_score, (seed_id,)) for seed_id, score in seed_scores.items()
    ]
    for _depth in range(hops):
        next_frontier: list[tuple[str, float, tuple[str, ...]]] = []
        for current_id, current_score, path in frontier:
            for neighbor_id, attributes in projection[current_id].items():
                if neighbor_id in path:
                    continue
                score = current_score * hop_decay * float(attributes["weight"])
                candidate_path = (*path, neighbor_id)
                if score > best_scores.get(neighbor_id, 0.0):
                    best_scores[neighbor_id] = score
                    best_paths[neighbor_id] = candidate_path
                next_frontier.append((neighbor_id, score, candidate_path))
        frontier = next_frontier
    return best_scores, best_paths


def edge_step(knowledge: KnowledgeGraph, left_id: str, right_id: str) -> str:
    candidates: list[tuple[float, str]] = []
    forward = cast(
        Mapping[object, Mapping[str, Any]],
        knowledge.graph.get_edge_data(left_id, right_id, default={}),
    )
    reverse = cast(
        Mapping[object, Mapping[str, Any]],
        knowledge.graph.get_edge_data(right_id, left_id, default={}),
    )
    for attributes in forward.values():
        kind = str(attributes["kind"])
        step = (
            f"-[{kind} {attributes['strength']:.2f}]-"
            if kind == "CO_CHANGES"
            else f"-[{kind} {attributes['strength']:.2f}]->"
        )
        candidates.append(
            (
                float(attributes["strength"]),
                step,
            )
        )
    for attributes in reverse.values():
        kind = str(attributes["kind"])
        step = (
            f"-[{kind} {attributes['strength']:.2f}]-"
            if kind == "CO_CHANGES"
            else f"<-[{kind} {attributes['strength']:.2f}]-"
        )
        candidates.append(
            (
                float(attributes["strength"]),
                step,
            )
        )
    return max(candidates, default=(0.0, "--"))[1]


def format_path(knowledge: KnowledgeGraph, path: Sequence[str]) -> str:
    if not path:
        return ""
    parts = [knowledge.nodes[path[0]].qualified_name]
    for left_id, right_id in itertools.pairwise(path):
        parts.extend(
            [
                edge_step(knowledge, left_id, right_id),
                knowledge.nodes[right_id].qualified_name,
            ]
        )
    return " ".join(parts)


def query_graph(
    knowledge: KnowledgeGraph,
    index: SearchIndex,
    query: str,
    *,
    seed_count: int = 6,
    hops: int = 2,
    related_count: int = 25,
    hop_decay: float = 0.85,
) -> QueryResult:
    if seed_count < 1 or hops < 1 or related_count < 1:
        raise ValueError("seed_count, hops, and related_count must all be positive")
    if not 0.0 < hop_decay <= 1.0:
        raise ValueError("hop_decay must be greater than 0.0 and at most 1.0")

    direct_scores = direct_query_scores(index, query)
    ranked_direct = sorted(
        direct_scores,
        key=lambda node_id: direct_scores[node_id],
        reverse=True,
    )
    anchors = tuple(ranked_direct[: min(seed_count, len(ranked_direct))])
    anchor_scores = {node_id: direct_scores[node_id] for node_id in anchors}

    projection = relationship_projection(knowledge)
    personalization_total = sum(anchor_scores.values())
    if personalization_total:
        personalization = {
            node_id: score / personalization_total for node_id, score in anchor_scores.items()
        }
    else:
        personalization = {node_id: 1.0 / len(anchors) for node_id in anchors}
    pagerank = nx.pagerank(
        projection,
        alpha=0.85,
        personalization=personalization,
        weight="weight",
    )

    relationship_scores, paths = strongest_paths(
        projection,
        anchor_scores,
        hops=hops,
        hop_decay=hop_decay,
    )
    for anchor in anchors:
        relationship_scores.pop(anchor, None)
        paths.pop(anchor, None)

    maximum_pagerank = max(pagerank.values(), default=1.0) or 1.0
    candidates = sorted(
        relationship_scores,
        key=lambda node_id: (
            relationship_scores[node_id],
            pagerank.get(node_id, 0.0),
        ),
        reverse=True,
    )[:related_count]

    relevant_rows = []
    for rank, node_id in enumerate(anchors, start=1):
        node = knowledge.nodes[node_id]
        relevant_rows.append(
            {
                "rank": rank,
                "node_id": node.id,
                "node": node.qualified_name,
                "kind": node.kind,
                "location": node.location,
                "direct_relevance": round(direct_scores[node_id], 4),
            }
        )

    related_rows = []
    for rank, node_id in enumerate(candidates, start=1):
        node = knowledge.nodes[node_id]
        related_rows.append(
            {
                "rank": rank,
                "node_id": node.id,
                "node": node.qualified_name,
                "kind": node.kind,
                "location": node.location,
                "relationship_strength": round(relationship_scores[node_id], 4),
                "pagerank": round(pagerank.get(node_id, 0.0) / maximum_pagerank, 4),
                "direct_relevance": round(direct_scores[node_id], 4),
                "strongest_path": format_path(knowledge, paths[node_id]),
            }
        )

    return QueryResult(
        query=query,
        anchors=anchors,
        relevant=pd.DataFrame(relevant_rows),
        related=pd.DataFrame(related_rows),
        direct_scores=direct_scores,
        relationship_scores=relationship_scores,
        pagerank_scores=pagerank,
        paths=paths,
    )

## Interactive visualization

The overview aggregates symbol relationships to files. The query graph includes the
selected anchors, returned related nodes, and the nodes on their strongest paths. Edge
width is proportional to strength. Hover text includes source location, confidence,
evidence count, and relation evidence.

In [19]:
NODE_COLORS: Mapping[str, str] = {
    "file": "#64748b",
    "class": "#a78bfa",
    "method": "#60a5fa",
    "function": "#34d399",
    "test": "#f472b6",
    "fixture": "#facc15",
}
EDGE_COLORS: Mapping[str, str] = {
    "CALLS": "#38bdf8",
    "CO_CHANGES": "#a78bfa",
    "CONTAINS": "#64748b",
    "IMPORTS": "#22d3ee",
    "INHERITS": "#f472b6",
    "INSTANTIATES": "#fb923c",
    "TESTS": "#facc15",
}
PLOTLY_CONFIG: Mapping[str, object] = {
    "displaylogo": False,
    "responsive": True,
    "scrollZoom": True,
}


@dataclass(frozen=True)
class VisualizationNode:
    id: str
    label: str
    group: str
    color: str
    size: float
    tooltip: str


@dataclass(frozen=True)
class VisualizationEdge:
    source_id: str
    target_id: str
    kind: str
    strength: float
    tooltip: str


def visualization_positions(
    nodes: Sequence[VisualizationNode],
    edges: Sequence[VisualizationEdge],
) -> dict[str, tuple[float, float]]:
    graph: nx.Graph[str] = nx.Graph()
    graph.add_nodes_from(node.id for node in nodes)
    for edge in edges:
        graph.add_edge(edge.source_id, edge.target_id, weight=edge.strength)
    positions = nx.spring_layout(
        graph,
        seed=42,
        weight="weight",
        iterations=100,
    )
    return {
        node_id: (float(coordinates[0]), float(coordinates[1]))
        for node_id, coordinates in positions.items()
    }


def build_plotly_network(
    nodes: Sequence[VisualizationNode],
    edges: Sequence[VisualizationEdge],
    *,
    title: str,
    height: int,
) -> go.Figure:
    positions = visualization_positions(nodes, edges)
    traces: list[Any] = []

    edge_groups: dict[tuple[str, int], list[VisualizationEdge]] = defaultdict(list)
    for edge in edges:
        strength_bucket = min(4, int(edge.strength * 4))
        edge_groups[(edge.kind, strength_bucket)].append(edge)

    legend_kinds: set[str] = set()
    for (kind, strength_bucket), grouped_edges in sorted(edge_groups.items()):
        x_values: list[float | None] = []
        y_values: list[float | None] = []
        for edge in grouped_edges:
            source_x, source_y = positions[edge.source_id]
            target_x, target_y = positions[edge.target_id]
            x_values.extend([source_x, target_x, None])
            y_values.extend([source_y, target_y, None])
        show_legend = kind not in legend_kinds
        legend_kinds.add(kind)
        traces.append(
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="lines",
                line={
                    "color": EDGE_COLORS.get(kind, "#94a3b8"),
                    "width": 0.75 + 1.25 * strength_bucket,
                },
                hoverinfo="skip",
                legendgroup=f"edge:{kind}",
                name=f"{kind} edge",
                showlegend=show_legend,
            )
        )

    traces.append(
        go.Scatter(
            x=[
                (positions[edge.source_id][0] + positions[edge.target_id][0]) / 2.0
                for edge in edges
            ],
            y=[
                (positions[edge.source_id][1] + positions[edge.target_id][1]) / 2.0
                for edge in edges
            ],
            mode="markers",
            marker={"color": "rgba(0,0,0,0.01)", "size": 12},
            hovertext=[edge.tooltip for edge in edges],
            hovertemplate="%{hovertext}<extra></extra>",
            name="relationship details",
            showlegend=False,
        )
    )

    nodes_by_group: dict[str, list[VisualizationNode]] = defaultdict(list)
    for node in nodes:
        nodes_by_group[node.group].append(node)
    for group, grouped_nodes in sorted(nodes_by_group.items()):
        traces.append(
            go.Scatter(
                x=[positions[node.id][0] for node in grouped_nodes],
                y=[positions[node.id][1] for node in grouped_nodes],
                mode="markers+text",
                text=[node.label for node in grouped_nodes],
                textposition="top center",
                textfont={"color": "#e2e8f0", "size": 10},
                marker={
                    "color": [node.color for node in grouped_nodes],
                    "line": {"color": "#e2e8f0", "width": 0.75},
                    "size": [node.size for node in grouped_nodes],
                },
                hovertext=[node.tooltip for node in grouped_nodes],
                hovertemplate="%{hovertext}<extra></extra>",
                name=group,
                legendgroup=f"node:{group}",
            )
        )

    figure = go.Figure(data=traces)
    figure.update_layout(
        title={"text": title, "x": 0.02},
        height=height,
        paper_bgcolor="#0f172a",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="closest",
        dragmode="pan",
        margin={"b": 20, "l": 20, "r": 20, "t": 60},
        legend={"bgcolor": "rgba(15,23,42,0.75)", "orientation": "h"},
        xaxis={"showgrid": False, "showticklabels": False, "zeroline": False},
        yaxis={"showgrid": False, "showticklabels": False, "zeroline": False},
        uirevision=title,
    )
    return figure


def display_network(figure: go.Figure, output_name: str) -> Path:
    output_path = ARTIFACT_DIR / output_name
    figure.write_html(
        str(output_path),
        include_plotlyjs=True,
        full_html=True,
        auto_open=False,
        config=dict(PLOTLY_CONFIG),
    )
    shell = get_ipython()
    if shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell":
        figure.show(renderer="plotly_mimetype", config=dict(PLOTLY_CONFIG))
    return output_path


def node_tooltip(node: CodeNode) -> str:
    signature = html.escape(node.signature or node.qualified_name)
    docstring = html.escape(node.docstring[:500])
    return (
        f"<b>{signature}</b><br>"
        f"{html.escape(node.location)}<br>"
        f"kind: {html.escape(node.kind)}<br>"
        f"{docstring}"
    )


def show_repository_overview(knowledge: KnowledgeGraph) -> Path:
    file_nodes = {node.id: node for node in knowledge.nodes.values() if node.kind == "file"}
    path_to_file = {node.path: node.id for node in file_nodes.values()}
    aggregated: dict[tuple[str, str, str], float] = {}
    for edge in knowledge.edges:
        source_file = path_to_file[knowledge.nodes[edge.source_id].path]
        target_file = path_to_file[knowledge.nodes[edge.target_id].path]
        if source_file == target_file:
            continue
        key = (source_file, target_file, edge.kind)
        current = aggregated.get(key, 0.0)
        aggregated[key] = 1.0 - (1.0 - current) * (1.0 - edge.strength)

    degree_counts: Counter[str] = Counter()
    overview_edges: list[VisualizationEdge] = []
    for (source_id, target_id, kind), strength in aggregated.items():
        degree_counts.update((source_id, target_id))
        source = file_nodes[source_id]
        target = file_nodes[target_id]
        connector = "--" if kind == "CO_CHANGES" else "->"
        overview_edges.append(
            VisualizationEdge(
                source_id=source_id,
                target_id=target_id,
                kind=kind,
                strength=strength,
                tooltip=(
                    f"<b>{kind}</b><br>{html.escape(source.path)} {connector} "
                    f"{html.escape(target.path)}<br>strength: {strength:.3f}"
                ),
            )
        )
    overview_nodes = [
        VisualizationNode(
            id=node.id,
            label=node.path,
            group="file",
            color=NODE_COLORS["file"],
            size=15.0 + 2.5 * math.log1p(degree_counts[node.id]),
            tooltip=node_tooltip(node),
        )
        for node in file_nodes.values()
    ]
    figure = build_plotly_network(
        overview_nodes,
        overview_edges,
        title="Repository file relationships",
        height=720,
    )
    return display_network(figure, "repository_overview.html")


def strongest_edge_between(
    knowledge: KnowledgeGraph,
    left_id: str,
    right_id: str,
) -> CodeEdge | None:
    matches = [
        edge for edge in knowledge.edges if {edge.source_id, edge.target_id} == {left_id, right_id}
    ]
    return max(matches, key=lambda edge: edge.strength, default=None)


def show_query_graph(
    knowledge: KnowledgeGraph,
    result: QueryResult,
) -> Path:
    selected: set[str] = set(result.anchors)
    related_ids = set(result.related["node_id"].tolist()) if not result.related.empty else set()
    selected.update(related_ids)
    for node_id in tuple(selected):
        selected.update(result.paths.get(node_id, ()))

    top_anchor = result.anchors[0] if result.anchors else ""
    query_nodes: list[VisualizationNode] = []
    for node_id in selected:
        node = knowledge.nodes[node_id]
        if node_id == top_anchor:
            color = "#fb7185"
            group = "top anchor"
        elif node_id in result.anchors:
            color = "#f59e0b"
            group = "anchor"
        else:
            color = NODE_COLORS.get(node.kind, "#38bdf8")
            group = node.kind
        score = max(
            result.direct_scores.get(node_id, 0.0),
            result.relationship_scores.get(node_id, 0.0),
        )
        query_nodes.append(
            VisualizationNode(
                id=node_id,
                label=node.qualified_name,
                group=group,
                color=color,
                size=14.0 + 24.0 * score,
                tooltip=(
                    f"{node_tooltip(node)}<br>direct relevance: "
                    f"{result.direct_scores.get(node_id, 0.0):.4f}<br>"
                    f"relationship strength: "
                    f"{result.relationship_scores.get(node_id, 0.0):.4f}"
                ),
            )
        )

    query_edges: list[VisualizationEdge] = []
    projection = relationship_projection(knowledge)
    for left_id, right_id in projection.subgraph(selected).edges:
        edge = strongest_edge_between(knowledge, left_id, right_id)
        if edge is None:
            continue
        source = knowledge.nodes[edge.source_id]
        target = knowledge.nodes[edge.target_id]
        connector = "--" if edge.kind == "CO_CHANGES" else "->"
        evidence = "<br>".join(html.escape(item) for item in edge.evidence)
        query_edges.append(
            VisualizationEdge(
                source_id=edge.source_id,
                target_id=edge.target_id,
                kind=edge.kind,
                strength=edge.strength,
                tooltip=(
                    f"<b>{edge.kind}</b><br>{html.escape(source.qualified_name)} "
                    f"{connector} {html.escape(target.qualified_name)}<br>"
                    f"strength: {edge.strength:.3f}<br>"
                    f"confidence: {edge.confidence:.3f}<br>"
                    f"evidence count: {edge.count}<br>{evidence}"
                ),
            )
        )
    figure = build_plotly_network(
        query_nodes,
        query_edges,
        title=f"Query relationships: {result.query}",
        height=760,
    )
    return display_network(figure, "query_graph.html")


overview_path = show_repository_overview(knowledge)
print(f"Standalone overview: {overview_path}")

Standalone overview: /home/user/projects/code-knowledge-graph/artifacts/repository_overview.html


## Run a query

Edit `QUERY` and rerun this cell. The first table shows text-relevant anchors. The second
table shows graph-related nodes ordered by strongest relationship path. Plotly emits a
portable notebook output and also writes `artifacts/query_graph.html` for standalone use.

In [20]:
QUERY = "where does a run pause for owner decisions and then resume?"

result = query_graph(
    knowledge,
    search_index,
    QUERY,
    seed_count=6,
    hops=2,
    related_count=25,
)

show(html_block(f"<h3>Relevant nodes for: {html.escape(QUERY)}</h3>"))
show(result.relevant)
show(html_block("<h3>Related nodes by relationship strength</h3>"))
show(result.related)
query_graph_path = show_query_graph(knowledge, result)
print(f"Standalone query graph: {query_graph_path}")

,rank,node_id,node,kind,location,direct_relevance
0,1,symbol:src/implicit_decision_gate/gate.py::sho...,show_payload,function,src/implicit_decision_gate/gate.py:376,0.1082
1,2,symbol:src/implicit_decision_gate/gate.py::Run...,RunStore.save,method,src/implicit_decision_gate/gate.py:171,0.0878
2,3,symbol:src/implicit_decision_gate/gate.py::ans...,answer_owner,function,src/implicit_decision_gate/gate.py:265,0.0867
3,4,symbol:src/implicit_decision_gate/gate.py::Run...,RunStore.load,method,src/implicit_decision_gate/gate.py:190,0.0807
4,5,symbol:src/implicit_decision_gate/agent.py::bu...,build_coding_prompt,function,src/implicit_decision_gate/agent.py:53,0.0799
5,6,symbol:src/implicit_decision_gate/orchestrator...,Orchestrator.resume,method,src/implicit_decision_gate/orchestrator.py:92,0.0791


,rank,node_id,node,kind,location,relationship_strength,pagerank,direct_relevance,strongest_path
0,1,symbol:src/implicit_decision_gate/gate.py::Gat...,GateError,class,src/implicit_decision_gate/gate.py:132,0.5290,0.7995,0.0300,answer_owner -[INSTANTIATES 0.78]-> GateError
1,2,symbol:src/implicit_decision_gate/gate.py::dec...,decision_request_payloads,function,src/implicit_decision_gate/gate.py:297,0.5253,0.4657,0.0759,show_payload -[CALLS 0.62]-> decision_request_...
2,3,symbol:src/implicit_decision_gate/gate.py::cov...,coverage_gap_payloads,function,src/implicit_decision_gate/gate.py:340,0.5253,0.3511,0.0364,show_payload -[CALLS 0.62]-> coverage_gap_payl...
3,4,symbol:src/implicit_decision_gate/gate.py::ren...,render_show,function,src/implicit_decision_gate/gate.py:413,0.5253,0.3203,0.0510,show_payload <-[CALLS 0.62]- render_show
4,5,symbol:src/implicit_decision_gate/agent.py::Ag...,AgentError,class,src/implicit_decision_gate/agent.py:19,0.4872,0.2257,0.0163,build_coding_prompt -[INSTANTIATES 0.78]-> Age...
5,6,symbol:src/implicit_decision_gate/gate.py::dec...,decision_requires_owner,function,src/implicit_decision_gate/gate.py:256,0.4782,0.2148,0.0491,answer_owner -[CALLS 0.70]-> decision_requires...
6,7,symbol:src/implicit_decision_gate/gate.py::Run...,RunStore.run_path,method,src/implicit_decision_gate/gate.py:143,0.4263,0.4173,0.0498,RunStore.save -[CALLS 0.62]-> RunStore.run_path
7,8,symbol:src/implicit_decision_gate/gate.py::utc...,utc_now,function,src/implicit_decision_gate/gate.py:53,0.4263,0.3255,0.0201,RunStore.save -[CALLS 0.62]-> utc_now
8,9,symbol:src/implicit_decision_gate/gate.py::Run...,RunStore.create,method,src/implicit_decision_gate/gate.py:150,0.4263,0.1902,0.0611,RunStore.save <-[CALLS 0.62]- RunStore.create
9,10,symbol:src/implicit_decision_gate/orchestrator...,Orchestrator.answer,method,src/implicit_decision_gate/orchestrator.py:78,0.4212,0.4086,0.0640,answer_owner <-[CALLS 0.62]- Orchestrator.answer


Standalone query graph: /home/user/projects/code-knowledge-graph/artifacts/query_graph.html


## Inspect a node and its strongest direct relationships

Use a node ID from `knowledge.nodes` or derive it from a query result. This table keeps
parallel relation types separate and sorts them by explicit edge strength.

In [21]:
def direct_relationships(
    knowledge: KnowledgeGraph,
    node_id: str,
) -> pd.DataFrame:
    if node_id not in knowledge.nodes:
        raise KeyError(f"Unknown node ID: {node_id}")
    rows = []
    for edge in knowledge.edges:
        if node_id not in {edge.source_id, edge.target_id}:
            continue
        outbound = edge.source_id == node_id
        related_id = edge.target_id if outbound else edge.source_id
        related = knowledge.nodes[related_id]
        rows.append(
            {
                "direction": "out" if outbound else "in",
                "relationship": edge.kind,
                "strength": round(edge.strength, 4),
                "confidence": round(edge.confidence, 4),
                "evidence_count": edge.count,
                "related_node": related.qualified_name,
                "location": related.location,
            }
        )
    return pd.DataFrame(rows).sort_values(
        ["strength", "relationship", "related_node"],
        ascending=[False, True, True],
        ignore_index=True,
    )


if result.anchors:
    selected_node_id = result.anchors[0]
    print(f"Selected: {selected_node_id}")
    show(direct_relationships(knowledge, selected_node_id))

Selected: symbol:src/implicit_decision_gate/gate.py::show_payload


,direction,relationship,strength,confidence,evidence_count,related_node,location
0,out,CALLS,0.618,0.95,1,coverage_gap_payloads,src/implicit_decision_gate/gate.py:340
1,out,CALLS,0.618,0.95,1,decision_request_payloads,src/implicit_decision_gate/gate.py:297
2,in,CALLS,0.618,0.95,1,render_show,src/implicit_decision_gate/gate.py:413
3,in,CONTAINS,0.450,1.00,1,implicit_decision_gate.gate,src/implicit_decision_gate/gate.py:1


## Next extensions

- Add an optional local sentence-transformer index and blend it with TF-IDF after the
  retrieval results have an evaluation set.
- Add Tree-sitter extractors for JavaScript, HTML, CSS, and SQL only when those files
  need symbol-level navigation. Their resolution confidence should remain separate.
- Persist nodes and edges as Parquet or SQLite when repeated indexing becomes slow.
- Add incremental invalidation keyed by file content hashes and the current Git commit.
- Build query fixtures with expected files and symbols, then tune edge priors and score
  fusion against measured recall instead of subjective examples.